# AI Avatars Bot - SadTalker Lip-Sync (Free Colab T4)

Run each cell top to bottom. This notebook:
1. Mounts your Google Drive.
2. Installs SadTalker (one-time per session).
3. Reads the newest audio file from the `AIAvatarsBot_Inbox` Drive folder.
4. Runs SadTalker against your avatar portrait image + that audio.
5. Writes the rendered talking-head mp4 to the `AIAvatarsBot_Outbox` Drive folder.

Colab's free tier does not support reliable unattended/scheduled execution, so you need to open this notebook and click Run All each time you want to render a video.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/OpenTalker/SadTalker.git
%cd SadTalker
!pip install -q -r requirements.txt
!bash scripts/download_models.sh

In [ ]:
import os

INBOX_DIR = '/content/drive/MyDrive/AIAvatarsBot_Inbox'
OUTBOX_DIR = '/content/drive/MyDrive/AIAvatarsBot_Outbox'
AVATAR_IMAGE_PATH = '/content/drive/MyDrive/AIAvatarsBot_Inbox/avatar.png'

os.makedirs(OUTBOX_DIR, exist_ok=True)

audio_files = [f for f in os.listdir(INBOX_DIR) if f.lower().endswith('.mp3')]
audio_files.sort(key=lambda f: os.path.getmtime(os.path.join(INBOX_DIR, f)), reverse=True)

if not audio_files:
    raise FileNotFoundError('No .mp3 files found in the inbox folder. Upload one with drive_sync.py first.')

latest_audio = os.path.join(INBOX_DIR, audio_files[0])
video_id = os.path.splitext(audio_files[0])[0]
print('Using audio file:', latest_audio)
print('Video id:', video_id)

if not os.path.exists(AVATAR_IMAGE_PATH):
    raise FileNotFoundError(
        'avatar.png not found in the inbox folder. Upload your one-time avatar portrait '
        'to Drive (see avatar_asset_setup.md) before running this cell.'
    )

In [ ]:
!python inference.py \
  --driven_audio "{latest_audio}" \
  --source_image "{AVATAR_IMAGE_PATH}" \
  --result_dir "/content/SadTalker/results" \
  --still \
  --preprocess full \
  --enhancer gfpgan

In [ ]:
import glob
import shutil

result_files = sorted(glob.glob('/content/SadTalker/results/**/*.mp4', recursive=True), key=os.path.getmtime)
if not result_files:
    raise FileNotFoundError('SadTalker did not produce an output video. Check the inference cell output above.')

latest_result = result_files[-1]
dest_path = os.path.join(OUTBOX_DIR, f'{video_id}_avatar.mp4')
shutil.copy(latest_result, dest_path)
print('Rendered video copied to:', dest_path)